# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset
using the `mlcroissant` library, referencing all entities by their `@id` fields for reproducibility and transparency.

### Dataset Source
The dataset is defined by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load metadata with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata

# Print basic metadata summary
print("Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", metadata.identifier)
print("Version:", metadata.version)
print("License:", metadata.license)
print("Spatial Coverage:", metadata.spatialCoverage)
print("Temporal Coverage:", metadata.temporalCoverage)
print("Keywords:", getattr(metadata, 'keywords', ''))

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.
All entity references are by `@id`.

Let's inspect the available record sets defined in the metadata.

In [ ]:
# List all RecordSet entities by '@id'
# The dataset may have multiple record sets defined as metadata.recordSet

record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if hasattr(rs, '@id'):
            record_sets.append(rs['@id'] if isinstance(rs, dict) else rs.@id)
else:
    print("No record sets found in metadata.")

# Print available record sets
print("Available RecordSets (@id):")
for rs_id in record_sets:
    print(rs_id)

if record_sets:
    # Print fields and columns for each record set by @id
    for rs in metadata.recordSet:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs.@id
        print("\nRecordSet @id:", rs_id)
        # Extract fields
        if hasattr(rs, 'field'):
            print("  Fields (@id):")
            for field in rs.field:
                print("    -", field['@id'] if isinstance(field, dict) else field.@id)
        # Extract columns
        if hasattr(rs, 'column'):
            print("  Columns (@id):")
            for column in rs.column:
                print("    -", column['@id'] if isinstance(column, dict) else column.@id)

## 3. Data Extraction
Load data from available record sets into Pandas DataFrames for further analysis.
Entities are referenced by their `@id` as per FAIR² standards.

In [ ]:
# Collect record set @id list
record_set_ids = record_sets

dataframes = {}

# Load data from each record set into a DataFrame
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with columns:")
        print(df.columns.tolist())
    except Exception as e:
        print(f"Could not load records for {record_set_id}:", str(e))

# Show a preview for the first record set (if any)
if record_set_ids:
    primary_record_set = record_set_ids[0]
    if primary_record_set in dataframes:
        print("\nPreview of the first 5 rows in DataFrame for", primary_record_set)
        display(dataframes[primary_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps.

We'll filter, normalize, and group using specific field and column `@id`s from the previous overview.

In [ ]:
# Choose a record set for EDA
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes.get(rs_id, pd.DataFrame())
else:
    rs_id = None
    df = pd.DataFrame()

# Identify candidate numeric and group fields by @id
numeric_field = None
group_field = None
if not df.empty:
    # Try to find numeric fields by inspecting data types
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Found numeric field: {numeric_field}")
    # Try to find a groupable field
    group_fields = [col for col in df.columns if df[col].dtype == 'object' and df[col].nunique() < 10]
    if group_fields:
        group_field = group_fields[0]
        print(f"Found group field: {group_field}")

if numeric_field and not df.empty:
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

    # Group if possible
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. All axes and legend entries reference the field `@id`.
We'll create a histogram of the numeric field and a boxplot grouped by the group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field} (@id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² dataset using `mlcroissant`, referencing all data entities by their `@id` fields. The approach ensures clear provenance and reproducibility.

- We accessed core metadata and inspected available record sets.
- Data was loaded and analyzed using `mlcroissant` and Pandas.
- Fields were filtered, normalized, grouped, and visualized to highlight key adoption predictors.
- The dataset supports research on knowledge adoption in rangeland management among marginalized communities in Northern Kenya.

For additional analysis and policy implications, refer to the dataset's documentation and cited sources. All processing steps can be repeated or extended using the `@id` referencing model.